# Задание 1. Сверточная сеть для собственных датасетов

Используя созданные датасеты, обучите свёрточную нейронную сеть для определения заболевания растения. Используя фиксированную (по свёрточным слоям) архитектуру модели, посмотрите на качество обучения при различных размерах входных изображений и опишите зависимость результата от размера входа.

In [11]:
# !wget https://storage.googleapis.com/ibeans/train.zip
# !wget https://storage.googleapis.com/ibeans/validation.zip
# !wget https://storage.googleapis.com/ibeans/test.zip

In [10]:
# !unzip train.zip
# !unzip validation.zip
# !unzip test.zip

In [12]:
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import ToTensor, Compose, ToPILImage
from torchvision.transforms import Normalize, Resize

from glob import glob
import torch
import os
from torch import nn

In [87]:
class BeanDataset(Dataset):
    def __init__(self, img_dir, transform = None):
        self.transform = transform
        # folder with images
        self.img_dir = img_dir
        # ordered list of all the images
        self.files = sorted(glob(f'{img_dir}/*/*.jpg'))
        class_names = sorted(os.listdir(img_dir))
        # map names to class idx
        self.class_dir = {name:idx for idx, name in enumerate(class_names)}

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        file_path = self.files[idx]
        image = Image.open(file_path)

        # applying transforms
        if self.transform:
          image = self.transform(image)

        # picking the penultimate folder name
        label_name = file_path.split('/')[-2]
        label = self.class_dir[label_name]
        return image, torch.tensor(label, dtype=torch.long)

In [114]:
transform = Compose([   Resize((256,256)), # You can change size
                        ToTensor(),
                        Normalize(
                            mean = [0.5183, 0.4845, 0.6570],
                            std = [0.2111, 0.2227, 0.2291]
                        )])

train_dataset = BeanDataset('./train/',transform)
valid_dataset = BeanDataset('./validation/',transform)
test_dataset = BeanDataset('./test/',transform)

In [115]:
batch_size = 50

In [116]:
train_loader = DataLoader(train_dataset, batch_size = 50)
test_loader = DataLoader(test_dataset, batch_size = 50)

In [117]:
val_loader = DataLoader(valid_dataset, batch_size = 50)

In [118]:
# Place your code here

class ConvNet(nn.Module):
  def __init__(self):
    super().__init__()
    self.pool = nn.MaxPool2d(2, 2)
    self.pool2 = nn.MaxPool2d(2, 2)
    self.act = nn.ReLU()
    self.conv = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(in_channels=3, out_channels=3, kernel_size=3, padding=1)
    self.fc = nn.Linear(3*64*64, 3)

  def forward(self, x):
    self.first = self.pool(self.act(self.conv(x)))
    self.second = self.pool2(self.act(self.conv2(self.first)))
    self.second  = self.second .view(x.size(0), -1)
    return self.fc(self.second)




In [119]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = ConvNet().to(device)

In [120]:
from tqdm import tqdm

In [121]:
from IPython.display import clear_output

In [122]:
opt = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()



for it in range(10):
  train_loss, val_loss = 0, 0
  model.train()
  for img, labels in train_loader:
    opt.zero_grad()
    out = model(img)
    batch_loss = criterion(out, labels)
    train_loss += batch_loss.item()
    batch_loss.backward()
    opt.step()
  print('\n')
  print(f'Train_loss per_epoch_№{it} = {train_loss/batch_size}')
  model.eval()
  with torch.no_grad():
    for img, labels in val_loader:
      out = model(img)
      batch_val_loss = criterion(out, labels)
      val_loss += batch_val_loss.item()
    print(f'Val_loss per_epoch_№{it} = {val_loss/batch_size}')








Train_loss per_epoch_№0 = 1.6539180646347813
Val_loss per_epoch_№0 = 0.11071446299552917


Train_loss per_epoch_№1 = 0.7641684019565582
Val_loss per_epoch_№1 = 0.06435294389724731


Train_loss per_epoch_№2 = 0.49143784046173095
Val_loss per_epoch_№2 = 0.06185158014297485


Train_loss per_epoch_№3 = 0.447722373008728
Val_loss per_epoch_№3 = 0.06041808128356933


Train_loss per_epoch_№4 = 0.43613823771476745
Val_loss per_epoch_№4 = 0.05879690647125244


Train_loss per_epoch_№5 = 0.43232624888420107
Val_loss per_epoch_№5 = 0.05738120913505554


Train_loss per_epoch_№6 = 0.43009849309921266
Val_loss per_epoch_№6 = 0.05632198691368103


Train_loss per_epoch_№7 = 0.4172314500808716
Val_loss per_epoch_№7 = 0.057490172386169436


Train_loss per_epoch_№8 = 0.437202479839325
Val_loss per_epoch_№8 = 0.05490788221359253


Train_loss per_epoch_№9 = 0.40474029898643493
Val_loss per_epoch_№9 = 0.05435643434524536


In [123]:
test_loss = 0
model.eval()
with torch.no_grad():
  for img, labels in test_loader:
    out = model(img)
    batch_test_loss = criterion(out, labels)
    test_loss += batch_test_loss.item()
  print(f'Test_loss = {test_loss/batch_size}')

Test_loss = 0.05647892832756043


In [ ]:
size = [256, 128, 64]
test_size_loss = [0.05647892832756043, 0.053742735385894774, 0.056929373741149904]

Напишите выводы:  
 ...